In [ ]:
#SETUP DOCKER+LANGFUSE

#!cd ~/Desktop/Mini-Project-LangFuse/langfuse
#!docker compose up -d
#http://localhost:3000/

In [ ]:
#SETUP LLM+LANGFUSE

#Import keys
import os
from dotenv import load_dotenv
load_dotenv()

#Set up the LLM client
from google import genai
from google.genai import types
gemini_client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

#Connect LangFuse
from langfuse import observe, Langfuse, get_client
langfuse = Langfuse()

In [ ]:
#TOOLS

#Tool 1: Drug interaction database
def check_drug_interactions(drug_a: str, drug_b: str) -> str:
    """Check if two drugs have any interactions or risks when taken together.
    
    Args:
        drug_a: First drug name.
        drug_b: Second drug name.
    """
    
    interactions = {
        ("ibuprofen", "lisinopril"): "WARNING: Ibuprofen may reduce the blood pressure lowering effect of lisinopril and increase risk of kidney damage.",
        ("aspirin", "warfarin"): "DANGER: Combining aspirin with warfarin significantly increases bleeding risk.",
        ("paracetamol", "ibuprofen"): "SAFE: These can generally be taken together at recommended doses.",
        ("metformin", "alcohol"): "WARNING: Alcohol can increase the risk of lactic acidosis with metformin.",
    }
    
    key = tuple(sorted([drug_a.lower(), drug_b.lower()]))
    
    return interactions.get(key, f"No known interaction data found for {drug_a} and {drug_b}.")

#Tool 2: Symptom checker
def check_symptoms(symptoms: str) -> str:
    """Look up possible conditions based on symptoms the user describes.
    
    Args:
        symptoms: The symptoms to look up.
    """
    
    symptom_map = {
        "chest pain": "Possible conditions: angina, heart attack, acid reflux, muscle strain. URGENT: If accompanied by shortness of breath, seek emergency care immediately.",
        "headache": "Possible conditions: tension headache, migraine, dehydration, sinusitis. Seek care if severe, sudden onset, or with fever/stiff neck.",
        "fever": "Possible conditions: viral infection, bacterial infection, flu. Seek care if above 39.5°C or lasting more than 3 days.",
    }
    
    for key, value in symptom_map.items():
        if key in symptoms.lower():
            return value
            
    return f"No data found for symptoms: {symptoms}. Please consult a healthcare professional."

#Tool 3: Medication info (mock)
def get_medication_info(medication: str) -> str:
    """Get basic information about a medication including uses, side effects, and dosage.
    
    Args:
        medication: The medication name.
    """
    
    meds = {
        "ibuprofen": "Type: NSAID. Used for: pain, inflammation, fever. Common side effects: stomach pain, nausea, dizziness. Max daily dose: 1200mg (OTC).",
        "paracetamol": "Type: Analgesic. Used for: pain, fever. Common side effects: rare at normal doses. Max daily dose: 4000mg. WARNING: liver damage risk if exceeded.",
        "lisinopril": "Type: ACE inhibitor. Used for: high blood pressure, heart failure. Common side effects: dry cough, dizziness, headache.",
        "metformin": "Type: Biguanide. Used for: type 2 diabetes. Common side effects: nausea, diarrhea, stomach pain.",
    }
    
    return meds.get(medication.lower(), f"No data found for {medication}.")

#Tool 4: Emergency protocol
def check_emergency(situation: str) -> str:
    """Check if a situation requires emergency medical attention and provide emergency protocols.
    
    Args:
        situation: Description of the emergency situation.
    """
    
    emergencies = {
        "choking": "EMERGENCY: 1) Call emergency services. 2) For conscious adult: perform Heimlich maneuver - stand behind, place fist above navel, thrust upward. 3) For unconscious: begin CPR.",
        "heart attack": "EMERGENCY: 1) Call emergency services immediately. 2) Have patient chew aspirin (if not allergic). 3) Keep patient calm and seated. 4) Be ready to perform CPR.",
        "stroke": "EMERGENCY: Use FAST method - Face drooping, Arm weakness, Speech difficulty, Time to call emergency services. Do NOT give medication.",
        "allergic reaction": "EMERGENCY: 1) Call emergency services. 2) Use epinephrine auto-injector if available. 3) Have patient lie down with legs elevated. 4) Monitor breathing.",
        "bleeding": "EMERGENCY: 1) Apply direct pressure with clean cloth. 2) Keep pressure for 10-15 minutes. 3) If severe or won't stop, call emergency services.",
    }
    
    for key, value in emergencies.items():
        if key in situation.lower():
            return value
    
    return f"No specific emergency protocol found for: {situation}. If in doubt, call emergency services immediately."

#Tool 5: Patient history lookup (mock)
def get_patient_history(patient_id: str) -> str:
    """Look up a patient's medical history including conditions, medications, and allergies.
    
    Args:
        patient_id: The patient's ID number.
    """
    
    patients = {
        "P001": "Name: John Smith. Age: 65. Conditions: hypertension, type 2 diabetes. Current medications: lisinopril 10mg, metformin 500mg. Allergies: penicillin.",
        "P002": "Name: Maria Garcia. Age: 42. Conditions: asthma, anxiety. Current medications: salbutamol inhaler, sertraline 50mg. Allergies: aspirin, sulfa drugs.",
        "P003": "Name: James Wilson. Age: 55. Conditions: high cholesterol, arthritis. Current medications: atorvastatin 20mg, ibuprofen 400mg. Allergies: none known.",
    }
    
    return patients.get(patient_id.upper(), f"No patient found with ID: {patient_id}")

#Tool 6: Appointment booking (mock)
def book_appointment(department: str, urgency: str) -> str:
    """Book a medical appointment in a specific department.
    
    Args:
        department: Medical department (e.g. cardiology, general, emergency, neurology).
        urgency: How urgent the appointment is (routine, urgent, emergency).
    """
    
    if urgency.lower() == "emergency":
        return f"EMERGENCY referral created for {department}. Patient should go to Emergency Department immediately."
    elif urgency.lower() == "urgent":
        return f"Urgent appointment booked in {department} for tomorrow morning. Patient will receive confirmation SMS."
    else:
        return f"Routine appointment booked in {department} for next week. Patient will receive confirmation SMS."

#Mapping
tool_functions = {
    "check_drug_interactions": check_drug_interactions,
    "check_symptoms": check_symptoms,
    "get_medication_info": get_medication_info,
    "check_emergency": check_emergency,
    "get_patient_history": get_patient_history,
    "book_appointment": book_appointment,
}

#Execute tool and trace it in LangFuse
@observe()
def execute_tool_traced(tool_name, tool_args):
    tool_func = tool_functions[tool_name]
    
    return tool_func(**tool_args)

In [ ]:
#AGENT

AGENT_SYSTEM_PROMPT = """You are a healthcare assistant with access to tools. 
Use the tools when you need to look up specific medical information.
If you can answer from general knowledge, do so directly.
Always recommend consulting a doctor for personalized advice.
Never diagnose or prescribe medication."""

@observe()
def healthcare_agent(user_question, max_iterations=8):    
    print(f"\n{'='*50}")
    print(f"User: {user_question}")
    print(f"{'='*50}")
    
    all_tools = [
        check_drug_interactions, check_symptoms, get_medication_info,
        check_emergency, get_patient_history, book_appointment
    ]
    
    contents = [
        types.Content(role="user", parts=[types.Part(text=user_question)])
    ]
    
    #ReAct loop (Think → Act → Observe → Repeat)
    for i in range(max_iterations):
        print(f"\n--- Iteration {i+1} ---")
        
        #THINK: Ask Gemini what to do (answer directly or use a tool?)
        response = gemini_client.models.generate_content(
            model="gemini-2.5-flash",
            contents=contents,
            config=types.GenerateContentConfig(
                system_instruction=AGENT_SYSTEM_PROMPT,
                tools=all_tools,
                automatic_function_calling=types.AutomaticFunctionCallingConfig(
                    disable=True
                )
            )
        )
        
        candidate = response.candidates[0]
        function_calls = response.function_calls
        
        #ACT + OBSERVE: If Gemini wants to use tool(s), execute them
        if function_calls:
            contents.append(candidate.content)
            
            for fc in function_calls:
                tool_name = fc.name
                tool_args = fc.args
                
                print(f"TOOL CALL: {tool_name}({tool_args})")
                
                #Execute the tool (traced in LangFuse via @observe)
                result = execute_tool_traced(tool_name, tool_args)
                print(f"TOOL RESULT: {result}")
                
                #Add tool result to conversation so Gemini can see it next iteration
                contents.append(
                    types.Content(
                        role="user",
                        parts=[types.Part.from_function_response(
                            name=tool_name,
                            response={"result": result}
                        )]
                    )
                )
        else:
            #No tool call → Gemini has enough info, return final answer
            answer = response.text
            print(f"\nFinal answer: {answer}")
            return answer
    
    return "Sorry, I couldn't find a complete answer."

In [ ]:
#TEST

#Test 1: Single tool
healthcare_agent("What are the side effects of metformin?")

#Test 2: Multi-tool — needs patient history + drug interaction check
healthcare_agent("Patient P001 wants to take ibuprofen for back pain. Is that safe given their current medications?")

#Test 3: Multi-tool — symptoms + emergency check + appointment
healthcare_agent("Patient P002 is having difficulty breathing and swelling in her throat. What should we do?")

#Test 4: Multi-tool — patient history + medication info
healthcare_agent("What medications is patient P003 currently on and what are their side effects?")

get_client().flush()